# 项目组织与打包

学习目标：把一组可导入的 Python 源码组织成可构建、可安装、带命令行入口的分发包，并区分代码检查、源码测试与安装检查各自证明的内容。

前置知识：模块与包、函数和类型标注、异常处理、文件路径、with、TOML、子进程参数列表和 pytest 的基本断言。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本例使用 Windows；pytest、build、Ruff、setuptools 与 wheel 的版本见 [requirements.txt](requirements.txt)。

构建在临时源码副本中完成，wheel 只安装到自动清理的临时目录；所有命令调用当前解释器，无需交互、网络下载或发布账号。

配套脚本：位于 [scripts/25-project-packaging/](scripts/25-project-packaging/)。

（1）[pyproject.toml](scripts/25-project-packaging/pyproject.toml)：分发元数据、构建后端、包发现和检查工具配置。

（2）[src/study\_minutes/](scripts/25-project-packaging/src/study_minutes/)：求和函数与命令行入口；[tests/](scripts/25-project-packaging/tests/)：公开行为与输入边界测试。

（3）[check\_installed.py](scripts/25-project-packaging/check_installed.py)：在指定安装目录核对导入位置、元数据与入口加载。

（4）[README.md](scripts/25-project-packaging/README.md)：包的用途与运行入口；[ci-example.yml](scripts/25-project-packaging/ci-example.yml)：未启用的持续集成配置示例。

## 1 从源码布局到两种包名称

本例把每次学习的非负整数分钟数相加：20、0、55 合计为 75。源码放在 src/study\_minutes/，测试放在 tests/，项目配置留在配套目录根部；这称为 src 布局。

src 布局让项目根目录与可导入源码分开，减少在项目根目录运行时意外导入源码副本的情况。这里的 src 不是导入包名。本章沿用模块与包的导入知识，重点检查这些文件如何进入安装产物。

| 名称 | 中文名称／含义 | 本例 |
| --- | --- | --- |
| import package | 导入包，供 Python 的 import 使用 | study\_minutes |
| distribution package | 分发包，供安装工具和分发元数据使用 | study-minutes-demo |

两种名称没有强制的一一对应规则；不能看到一个 import 名称就猜测应安装哪个同名项目。

In [1]:
from pathlib import Path

project_dir = Path("scripts/25-project-packaging").resolve()
assert (project_dir / "pyproject.toml").is_file()
for path in sorted(project_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(project_dir).as_posix())
# 包代码应位于 src/study_minutes/，tests 和工具脚本不在导入包中。

check_installed.py
ci-example.yml
pyproject.toml
README.md
src/study_minutes/__init__.py
src/study_minutes/cli.py
tests/test_minutes.py


## 2 统一环境与不同用途的依赖

本项目使用同一个 Conda 环境，安装操作按开始使用文档执行。requirements.txt 记录本技术目录需要准备的工具；示例包的 pyproject.toml 则描述它自己的构建条件和运行条件。

| 配置位置 | 中文名称／含义 |
| --- | --- |
| build-system.requires | 构建依赖，用于生成分发文件 |
| project.dependencies | 运行依赖，供安装后的业务代码使用 |
| requirements.txt | 本技术目录的环境准备清单，包含教学与开发工具 |

本例没有第三方运行依赖，因此 dependencies 是空列表。pytest 和 Ruff 用于开发检查，不应仅因为开发时用到了它们，就让最终用户安装它们。

固定版本能减少依赖升级引入的变化；要进一步重建环境，还应控制间接依赖、Python 与平台条件，并按需要核对分发文件哈希。本章固定已准备好的工具版本，不把几项直接依赖的版本号当作完整环境锁定。

In [2]:
import tomllib

with (project_dir / "pyproject.toml").open("rb") as stream:
    pyproject = tomllib.load(stream)

print(pyproject["build-system"]["requires"])  # ['setuptools==83.0.0', 'wheel==0.47.0']。
print(pyproject["project"]["dependencies"])  # []：无第三方运行依赖。
assert pyproject["build-system"]["requires"] == [
    "setuptools==83.0.0", "wheel==0.47.0"
]
# tomllib 读取 TOML；清单中的构建工具不会自动成为运行依赖。

['setuptools==83.0.0', 'wheel==0.47.0']
[]


## 3 pyproject.toml 的元数据与命令入口

pyproject.toml 集中保存构建配置、项目元数据和工具配置。下面的方括号表示 TOML 表名；build-system、project、tool 是各自负责不同内容的表。

| 配置名称 | 中文名称／含义 |
| --- | --- |
| build-system.build-backend | 构建后端的导入位置，本例为 setuptools.build\_meta |
| project.name | 分发包名称 |
| project.version | 当前分发版本，本例为 0.1.0 |
| project.readme | 作为项目说明的文件 |
| project.requires-python | 允许安装的 Python 版本范围 |
| project.scripts | 安装时生成的命令与 Python 入口的对应关系 |
| tool.setuptools.packages.find | setuptools 的包查找设置 |
| tool.ruff | Ruff 的检查与格式配置 |

requires-python 写为 >=3.12，声明最低 Python 版本；它不会测试代码在所有更高版本上是否可用。Ruff 的 py312 也只是工具的目标版本，不能代替对应解释器上的实际测试。项目尚未选定许可证，所以配置不填写 license。

study-minutes = "study\_minutes.cli:main" 把命令名映射到模块 study\_minutes.cli 中的 main 函数。冒号左侧是模块路径，右侧是可调用对象名称；安装工具生成包装程序，调用入口并把返回值交给退出状态处理。

In [3]:
print((project_dir / "pyproject.toml").read_text(encoding="utf-8"))
# where = ["src"] 从 src 查包；include 只选择 study_minutes。
# main 可无参数调用，由 argparse 从进程参数读取输入。

[build-system]
requires = ["setuptools==83.0.0", "wheel==0.47.0"]
build-backend = "setuptools.build_meta"

[project]
name = "study-minutes-demo"
version = "0.1.0"
description = "统计学习分钟数的本地打包示例"
readme = "README.md"
requires-python = ">=3.12"
dependencies = []

[project.scripts]
study-minutes = "study_minutes.cli:main"

[tool.setuptools.packages.find]
where = ["src"]
include = ["study_minutes"]
namespaces = false

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = ["--import-mode=importlib"]

[tool.ruff]
target-version = "py312"
line-length = 79

[tool.ruff.lint]
select = ["E", "F", "I"]



## 4 先观察源码提供的行为

包的 total\_minutes 函数接收分钟数列表，空列表返回零，拒绝负数、布尔值与非整数。cli.main 使用 argparse 解析至少一个整数，成功打印合计并返回 0；用法错误退出为 2。

本节在子进程中显式设置 PYTHONPATH 指向 src，仅用于观察源码。传给 subprocess.run 的 env 是环境映射副本，保留原有系统变量；它不会修改 Notebook 进程的环境或搜索路径。后面安装检查会改为指向安装目标。

In [4]:
import os
import subprocess
import sys

child_env = os.environ.copy()
child_env.update({
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTHONUTF8": "1",
    "PYTHONIOENCODING": "utf-8",
    "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
})
source_env = {**child_env, "PYTHONPATH": str(project_dir / "src")}
result = subprocess.run(
    [sys.executable, "-B", "-m", "study_minutes.cli", "20", "0", "55"],
    cwd=project_dir, env=source_env, capture_output=True,
    encoding="utf-8", check=True, timeout=30,
)
print(result.stdout, end="")  # 合计：75 分钟。
assert result.stdout == "合计：75 分钟\n"
assert result.stderr == ""
# -B 与对应环境变量避免生成字节码缓存；子进程结束后即可释放资源。

合计：75 分钟


## 5 PEP 8 与实际代码检查

### 5.1 风格约定和工具检查各有范围

PEP 8 建议使用 4 个空格缩进、按标准库／第三方／本项目组织导入，并使用小写加下划线的函数名。它通常建议代码行不超过 79 个字符，注释和文档字符串不超过 72 个字符；本例 Ruff 的代码行宽设为 79。

Ruff check 按启用的规则检查代码，本例选择 E、F、I，分别覆盖 pycodestyle、Pyflakes 与导入排序规则。Ruff format --check 只核对格式，不改文件。两者通过都不表示业务逻辑正确，也不等于覆盖了 PEP 8 的全部文字建议。

配套文件头保留完整的可复制运行命令，在模块 docstring 的结束处用 noqa: E501 仅豁免该字符串的行宽检查，其余代码仍按 79 字符检查。

下面两个命令都以配套目录为工作目录，--no-cache 避免留下 Ruff 缓存。

In [5]:
for arguments in (
    ["check", "--no-cache", "."],
    ["format", "--check", "--no-cache", "."],
):
    result = subprocess.run(
        [sys.executable, "-B", "-m", "ruff", *arguments],
        cwd=project_dir, env=child_env, capture_output=True,
        encoding="utf-8", check=True, timeout=30,
    )
    print(result.stdout, end="")  # 依次为 All checks passed!、5 files already formatted。
# 两项都必须返回 0；不会自动改写源码，也不会把格式通过当作测试通过。

All checks passed!


5 files already formatted


### 5.2 用一个失败示例理解退出状态

Ruff check 在发现未修复的问题时返回 1；配置或命令等异常则返回 2。本例把含未使用导入的短代码从标准输入传入，只检查 F401，观察真实诊断。

这里不使用 check=True，因为返回 1 是刻意设计的观察目标；必须断言恰好为 1，并核对具体规则，不能把任意非零退出都当作预期失败。

In [6]:
result = subprocess.run(
    [sys.executable, "-B", "-m", "ruff", "check", "--no-cache",
     "--select", "F401", "--stdin-filename", "example.py", "-"],
    input="import math\n", cwd=project_dir, env=child_env,
    capture_output=True, encoding="utf-8", timeout=30,
)
assert result.returncode == 1, (result.returncode, result.stderr)
assert "F401" in result.stdout
print(result.stdout, end="")
# 诊断应指向 math 的未使用导入；输入只在内存中，没有创建错误源码文件。

F401 [*] `math` imported but unused
 --> example.py:1:8
  |
1 | import math
  |        ^^^^
help: Remove unused import: `math`
  |
  - import math
  |

Found 1 error.
[*] 1 fixable with the `--fix` option.


## 6 源码测试能证明什么

pytest 默认按命名规则发现测试。配套 tests/test\_minutes.py 使用参数化、预期异常和输出捕获，检查空输入、零值、正常合计、非法类型与 CLI 的退出状态。

本例使用 importlib 导入模式，并在子进程中显式把 src 加入搜索路径。这样的测试观察的是源码行为；如果构建配置漏掉包文件，源码测试仍可能通过。因此，后面还要检查实际 wheel 和安装结果。

-p no:cacheprovider 关闭测试缓存插件；环境变量 PYTEST\_DISABLE\_PLUGIN\_AUTOLOAD 关闭其他已安装插件的自动加载，避免共享环境中的无关插件影响本例。

In [7]:
result = subprocess.run(
    [sys.executable, "-B", "-m", "pytest", "-q", "-p", "no:cacheprovider"],
    cwd=project_dir, env=source_env, capture_output=True,
    encoding="utf-8", check=True, timeout=60,
)
print(result.stdout, end="")
# 本套参数组合应形成 12 个测试用例；全部通过仍不能替代安装检查。

............                                                             [100%]
12 passed in 0.08s


## 7 构建前端、后端与两类分发文件

构建前端负责读取配置并调用后端；后端负责把项目文件组织为分发文件。本例用 python -m build 调用前端，用 setuptools.build\_meta 作为后端。

源码能运行，不等于安装包一定包含相同文件。沿构建与安装的链路检查每一份产物，才能定位“源码测试通过、安装后缺文件”的差别。

![源码、源码分发包与已构建分发包](image/illustration/25-01-build-artifacts.svg)

图示：本章默认构建路径。wheel 是已构建分发格式，纯 Python 项目同样会生成 wheel。

| 名称 | 中文名称／含义 |
| --- | --- |
| build frontend | 构建前端，本例为 build |
| build backend | 构建后端，本例为 setuptools |
| wheel | 已构建分发格式，扩展名为 .whl，可包含纯 Python 代码 |
| sdist | 源码分发包，本例为 .tar.gz，包含后续构建所需源码和元数据 |

build 默认先生成 sdist，再从该 sdist 生成 wheel。--no-isolation 关闭另建隔离构建环境，要求构建依赖已在当前解释器中准备好；本例保留依赖检查，不下载依赖。

构建可能生成 build 和 egg-info 等中间目录，所以先复制配套目录到 TemporaryDirectory。退出 with 时删除副本和分发文件，只把这两个小文件的字节保留在内存中，供后续检查。

下面先在临时副本构建两个归档；随后检查归档内容，再安装 wheel 验证入口，分别覆盖图中的不同边界。

In [8]:
import shutil
import tempfile

with tempfile.TemporaryDirectory() as directory:
    build_root = Path(directory)
    # 构建中间文件只写入源码副本，正式示例目录保持可重复使用。
    build_copy = shutil.copytree(project_dir, build_root / "project")
    output_dir = build_root / "artifacts"
    result = subprocess.run(
        [sys.executable, "-B", "-m", "build", "--no-isolation",
         "--outdir", str(output_dir), str(build_copy)],
        cwd=build_root, env=child_env, capture_output=True,
        encoding="utf-8", check=True, timeout=120,
    )
    (wheel_path,) = output_dir.glob("*.whl")
    (sdist_path,) = output_dir.glob("*.tar.gz")
    wheel_name = wheel_path.name
    sdist_name = sdist_path.name
    # 离开临时目录前保留两个小型归档的字节，供后续检查和安装。
    wheel_bytes = wheel_path.read_bytes()
    sdist_bytes = sdist_path.read_bytes()
    print(wheel_name)  # study_minutes_demo-0.1.0-py3-none-any.whl。
    print(sdist_name)  # study_minutes_demo-0.1.0.tar.gz。

assert not build_root.exists()
# 应得到 0.1.0 的 wheel 和 sdist；源码目录没有承担构建写入。

study_minutes_demo-0.1.0-py3-none-any.whl
study_minutes_demo-0.1.0.tar.gz


## 8 检查分发文件中的路径与元数据

wheel 是 ZIP 格式，包文件位于安装所需的相对位置，并带有 .dist-info 元数据目录。源码的 src 只是布局层，通常不会成为 wheel 中的包路径。

本例 wheel 名称中的 py3、none、any 分别表示 Python 3、没有特定 ABI 要求、没有特定平台要求；ABI 指二进制接口。它仍受 Requires-Python 的版本条件约束，不能仅凭 py3 判断 Python 3.11 也可安装。

sdist 内有一个顶层目录，其中包含 pyproject.toml、源码与 PKG-INFO。本节只读取自己刚构建的归档，不把文件解压到仓库。

In [9]:
import email.parser
import io
import tarfile
import zipfile

# 先读取 wheel 的安装布局，再读取 sdist 的源码布局；本单元不解压。
with zipfile.ZipFile(io.BytesIO(wheel_bytes)) as archive:
    wheel_members = archive.namelist()
    assert "study_minutes/__init__.py" in wheel_members
    assert "study_minutes/cli.py" in wheel_members
    assert not any(name.startswith("src/") for name in wheel_members)
    (metadata_name,) = [
        name for name in wheel_members if name.endswith(".dist-info/METADATA")
    ]
    wheel_metadata = email.parser.BytesParser().parsebytes(
        archive.read(metadata_name)
    )
    print(wheel_metadata["Name"], wheel_metadata["Version"])  # study-minutes-demo 0.1.0。
    print(wheel_metadata["Requires-Python"])  # >=3.12。
    assert wheel_metadata.get_all("Requires-Dist", []) == []

with tarfile.open(fileobj=io.BytesIO(sdist_bytes), mode="r:gz") as archive:
    sdist_members = archive.getnames()
    root = sdist_name.removesuffix(".tar.gz")
    for relative in ("pyproject.toml", "PKG-INFO", "src/study_minutes/cli.py"):
        assert f"{root}/{relative}" in sdist_members
        print(relative)  # 依次列出 pyproject.toml、PKG-INFO、src/study_minutes/cli.py。
# 这里检查实际归档，而不是根据源码文件存在就推测它们已经被打包。

study-minutes-demo 0.1.0
>=3.12
pyproject.toml
PKG-INFO
src/study_minutes/cli.py


## 9 安装 wheel 并运行安装后的入口

pip install --target 把包安装到指定目录。这里传入刚构建的本地 wheel，并使用 --no-deps、--no-index，分别禁止安装运行依赖和查询包索引；本例 dependencies 为空，所以无需额外依赖解析。

安装后在源码目录外启动新进程，并只把安装目标写入该子进程的 PYTHONPATH。配套检查脚本核对导入位置、分发元数据与输入边界，再从分发包的 console\_scripts 入口记录加载 main；仅手写 import 入口函数不能验证入口是否登记正确。

最后实际执行 Windows 安装包装程序 study-minutes.exe，核对标准输出与退出状态。--target 是安装位置，不会创建新的 Python 环境，也不会自动把目录加进 Notebook 的搜索路径或系统 PATH。

In [10]:
with tempfile.TemporaryDirectory() as directory:
    install_root = Path(directory)
    local_wheel = install_root / wheel_name
    local_wheel.write_bytes(wheel_bytes)
    target = install_root / "installed"
    outside_dir = install_root / "outside"
    outside_dir.mkdir()

    # 1. 真正安装本地 wheel；额外关闭 pip 缓存、字节码和版本联网检查。
    result = subprocess.run(
        [sys.executable, "-B", "-m", "pip", "install", str(local_wheel),
         "--target", str(target), "--no-deps", "--no-index", "--no-compile",
         "--no-cache-dir", "--disable-pip-version-check"],
        cwd=outside_dir, env=child_env, capture_output=True,
        encoding="utf-8", check=True, timeout=60,
    )
    installed_env = {**child_env, "PYTHONPATH": str(target)}
    check_script = shutil.copy2(
        project_dir / "check_installed.py", outside_dir / "check_installed.py"
    )
    result = subprocess.run(
        [sys.executable, "-B", str(check_script), str(target)],
        cwd=outside_dir, env=installed_env, capture_output=True,
        encoding="utf-8", check=True, timeout=30,
    )
    print(result.stdout, end="")  # 安装位置、元数据、输入边界与入口加载均通过。

    # 2. Windows 包装程序由安装工具生成；绝对路径调用无需修改 PATH。
    (wrapper,) = target.rglob("study-minutes.exe")
    print(wrapper.relative_to(target).as_posix())  # 本章 Windows 安装为 bin/study-minutes.exe。
    result = subprocess.run(
        [str(wrapper), "20", "0", "55"],
        cwd=outside_dir, env=installed_env, capture_output=True,
        encoding="utf-8", check=True, timeout=30,
    )
    assert result.stdout == "合计：75 分钟\n"
    assert result.stderr == ""
    print(result.stdout, end="")  # 合计：75 分钟。
    invalid = subprocess.run(
        [str(wrapper), "-1"], cwd=outside_dir, env=installed_env,
        capture_output=True, encoding="utf-8", timeout=30,
    )
    assert invalid.returncode == 2, (invalid.returncode, invalid.stderr)
    assert "学习分钟数不能为负数" in invalid.stderr
    assert invalid.stdout == ""
    print("负数输入退出状态：", invalid.returncode)  # 2。

assert not install_root.exists()
# 安装文件、包装程序及检查脚本副本都已删除，无需卸载共享环境中的包。

安装位置、元数据、输入边界与入口加载均通过
bin/study-minutes.exe


合计：75 分钟
负数输入退出状态： 2


## 10 持续集成与敏感配置

持续集成（continuous integration，CI）让预先配置的检查在工作流事件触发后运行。配套 YAML 示例把源码检查、格式检查、pytest 分成步骤；这些步骤使用进程的退出状态报告成败。

本例假定配套目录成为独立仓库根目录，选择已经准备好共享 Conda 环境的自托管 Windows runner；runner 是执行工作流的机器。PYTHON\_EXE 指向该机器的解释器，搬到其他机器时需调整。actions/checkout v6 使用的运行时要求 runner 至少为 2.327.1。

workflow\_dispatch 表示手动触发。GitHub 要求工作流放在仓库 .github/workflows/ 下，且手动事件要求文件位于默认分支；当前文件留在配套目录，因此没有启用远程工作流。这个最小示例只列源码检查，构建与安装链路已在前文单独执行。

普通参数可进入项目配置；令牌等敏感值不写进 pyproject.toml、源码或构建包。需要认证的 CI 应通过服务提供的 secrets 机制注入，并避免在命令参数或输出中暴露。本例不需要凭证，也不执行发布。

In [11]:
ci_text = (project_dir / "ci-example.yml").read_text(encoding="utf-8")
print(ci_text)  # 显示 YAML：依次声明代码检查、格式检查和测试三个步骤。
# 这里只阅读配置；打印 YAML 不代表 CI 已触发或远程检查已经通过。

# 阅读示例：假定此配套目录成为独立仓库根目录，未在本仓库启用。
# 自托管 Windows runner 必须已准备统一 Conda 环境和本章所需工具。
name: Python source checks
on:
  workflow_dispatch:
permissions:
  contents: read
jobs:
  checks:
    runs-on: [self-hosted, Windows]
    defaults:
      run:
        shell: powershell
    env:
      PYTHON_EXE: C:/Users/ZHUANG/miniconda3/envs/hands-on-computing/python.exe
      PYTHONDONTWRITEBYTECODE: "1"
      PYTHONUTF8: "1"
      PYTHONIOENCODING: utf-8
      PYTEST_DISABLE_PLUGIN_AUTOLOAD: "1"
    steps:
      - uses: actions/checkout@v6
      - name: Check code
        run: '& $env:PYTHON_EXE -B -m ruff check --no-cache .'
      - name: Check formatting
        run: '& $env:PYTHON_EXE -B -m ruff format --check --no-cache .'
      - name: Test source
        run: |
          $env:PYTHONPATH = Join-Path $PWD "src"
          & $env:PYTHON_EXE -B -m pytest -q -p no:cacheprovider



## 本章小结

（1）src 布局区分项目根目录与导入源码；导入包名、分发包名和命令名各有用途。

（2）环境清单、构建依赖与运行依赖分别维护自己的职责；工具版本固定有助于复现，但不等于完整锁定所有运行条件。

（3）PEP 8 给出风格建议，Ruff 检查规则与格式，pytest 检查选定行为。构建前端调用后端生成 sdist 和 wheel。

（4）安装检查应离开源码目录，核对实际导入位置、元数据与生成的入口。版本条件表达兼容声明，实际兼容性仍需在目标解释器上运行检查。

自查：源码测试通过后，如果 wheel 漏掉包或命令入口，哪一步才能发现问题？

## 练习

（1）先预测下面三个布尔结果，再运行核对。分别说明分发包名、入口映射和 wheel 内路径的含义；核对标准是预测与实际值一致，并能解释 src 为什么没有保留在包路径中。

In [12]:
print(pyproject["project"]["name"] == "study_minutes")
entry_mapping = pyproject["project"]["scripts"]["study-minutes"]
print(entry_mapping == "study_minutes.cli:main")
print("src/study_minutes/__init__.py" in wheel_members)
# 先写预测，再核对本章配置与实际归档成员；不要修改原文件来匹配预测。

False
True
False


（2）复制配套目录到新的 TemporaryDirectory，只把 project.version 从 0.1.0 改为 0.2.0，再用相同的 --no-isolation 流程构建。检查 wheel 和 sdist 文件名包含 0.2.0，wheel 的 Version 也为 0.2.0，而 Name、Requires-Python 和命令入口保持原值。

离开 with 后确认临时目录不存在、原 pyproject.toml 仍为 0.1.0。check\_installed.py 专门检查原始 0.1.0，本题自行读取新归档的元数据，不直接套用它的版本断言。

In [13]:
exercise_version = "0.2.0"
# 在临时源码副本中修改版本，构建后读取 METADATA 与 entry_points.txt。
# 检查新版本的两个产物，退出 with 后核对清理和原文件版本。

（3）使用本章 wheel 字节，在新的临时目录安装一次。保持工作目录位于源码之外，PYTHONPATH 仅在子进程中指向安装目标，实际运行 Windows 包装程序。

输入 0 45 时，标准输出应为“合计：45 分钟”并换行，退出状态为 0；没有参数、输入 -1 或 1.5 时，标准输出为空，标准错误包含用法错误，退出状态为 2。检查输出没有 Traceback，并确认临时目录已删除、Notebook 的环境变量未改变。

In [14]:
exercise_arguments = [["0", "45"], [], ["-1"], ["1.5"]]
# 在同一个 with 中完成 wheel 写入、pip 安装、四次子进程调用与断言。
# 每次检查明确的退出状态；不能用“非零即通过”代替错误边界核对。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [TOML 二进制流读取](https://docs.python.org/3.12/library/tomllib.html#tomllib.load)；[子进程参数、工作目录、环境映射与超时](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)；[PYTHONPATH](https://docs.python.org/3.12/using/cmdline.html#envvar-PYTHONPATH)、[关闭字节码写入](https://docs.python.org/3.12/using/cmdline.html#envvar-PYTHONDONTWRITEBYTECODE)；[临时目录生命周期](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)；[ZIP 成员列表](https://docs.python.org/3.12/library/zipfile.html#zipfile.ZipFile.namelist)、[ZIP 字节读取](https://docs.python.org/3.12/library/zipfile.html#zipfile.ZipFile.read)、[tar 成员列表](https://docs.python.org/3.12/library/tarfile.html#tarfile.TarFile.getnames)、[元数据字节解析](https://docs.python.org/3.12/library/email.parser.html#email.parser.BytesParser.parsebytes)；[分发元数据](https://docs.python.org/3.12/library/importlib.metadata.html#distribution-metadata)、[入口查询与加载](https://docs.python.org/3.12/library/importlib.metadata.html#entry-points)；[argparse 参数数量](https://docs.python.org/3.12/library/argparse.html#nargs)、[参数类型转换](https://docs.python.org/3.12/library/argparse.html#type)、[用法错误与退出状态 2](https://docs.python.org/3.12/library/argparse.html#argparse.ArgumentParser.error)。 |
| PyPA 打包指南与规范 | [src 布局与导入位置](https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/)；[分发包与导入包名称](https://packaging.python.org/en/latest/discussions/distribution-package-vs-import-package/#what-are-the-links-between-distribution-packages-and-import-packages)；[构建后端声明](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#declaring-the-build-backend)、[依赖](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#dependencies-and-requirements)、[版本兼容声明](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#requires-python)、[命令入口](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#creating-executable-scripts)；[wheel 格式与文件名](https://packaging.python.org/en/latest/specifications/binary-distribution-format/#file-format)、[Python 与 ABI、平台标签](https://packaging.python.org/en/latest/specifications/platform-compatibility-tags/)、[源码分发格式](https://packaging.python.org/en/latest/specifications/source-distribution-format/#source-distribution-file-format)。 |
| Python PEP 文档 | PEP 8 的[缩进](https://peps.python.org/pep-0008/#indentation)、[最大行长](https://peps.python.org/pep-0008/#maximum-line-length)、[导入分组](https://peps.python.org/pep-0008/#imports)、[函数与变量命名](https://peps.python.org/pep-0008/#function-and-variable-names)。 |
| GitHub 上的维护者源码 | setuptools 83.0.0 的 [pyproject 配置与包查找](https://github.com/pypa/setuptools/blob/v83.0.0/docs/userguide/pyproject_config.rst)，定位 example-pyproject-config 与 setuptools-table；[命令入口](https://github.com/pypa/setuptools/blob/v83.0.0/docs/userguide/entry_point.rst)，定位 Console Scripts；actions/checkout v6 的 [README 与 runner 版本要求](https://github.com/actions/checkout/blob/v6/README.md#whats-new)。 |
| Ruff 官方文档 | 本章执行 Ruff 0.16.7；[配置文件与作用范围](https://docs.astral.sh/ruff/configuration/#config-file-discovery)、[规则选择](https://docs.astral.sh/ruff/linter/#rule-selection)、[多行文档字符串的行宽豁免](https://docs.astral.sh/ruff/rules/line-too-long/#error-suppression)、[检查退出状态](https://docs.astral.sh/ruff/linter/#exit-codes)、[格式检查](https://docs.astral.sh/ruff/formatter/#ruff-format)。 |
| pytest 官方文档 | 本章执行 pytest 9.1.1；[src 布局、源码测试与 importlib 导入模式](https://docs.pytest.org/en/stable/explanation/goodpractices.html#tests-outside-application-code)、[禁用插件](https://docs.pytest.org/en/stable/how-to/usage.html#disabling-plugins)、[关闭插件自动加载](https://docs.pytest.org/en/stable/reference/reference.html#envvar-PYTEST_DISABLE_PLUGIN_AUTOLOAD)；[参数化](https://docs.pytest.org/en/stable/how-to/parametrize.html#pytest-mark-parametrize-parametrizing-test-functions)、[预期异常](https://docs.pytest.org/en/stable/how-to/assert.html#assertions-about-expected-exceptions)、[捕获标准输出与标准错误](https://docs.pytest.org/en/stable/how-to/capture-stdout-stderr.html#accessing-captured-output-from-a-test-function)。 |
| build 官方文档（1.6.1） | [构建前端职责](https://build.pypa.io/en/stable/#build)；[默认先构建 sdist 再构建 wheel、输出目录及命令选项](https://build.pypa.io/en/stable/reference/cli.html#python--m-build)、[关闭隔离后的依赖检查](https://build.pypa.io/en/stable/reference/cli.html#dependency-check)。 |
| pip 官方文档 | [目标目录安装](https://pip.pypa.io/en/stable/cli/pip_install/#install-target)、[不安装依赖](https://pip.pypa.io/en/stable/cli/pip_install/#install-no-deps)、[不查询索引](https://pip.pypa.io/en/stable/cli/pip_install/#install-no-index)、[关闭安装时编译](https://pip.pypa.io/en/stable/cli/pip_install/#install-no-compile)；[固定直接及间接依赖](https://pip.pypa.io/en/stable/topics/repeatable-installs/#pinning-the-package-versions)、[哈希核对](https://pip.pypa.io/en/stable/topics/repeatable-installs/#hash-checking)。 |
| GitHub Docs | [工作流文件位置](https://docs.github.com/en/actions/reference/workflows-and-actions/workflow-syntax#about-yaml-syntax-for-workflows)、[手动触发与默认分支](https://docs.github.com/en/actions/reference/workflows-and-actions/workflow-syntax#onworkflow_dispatch)、[runner 标签选择](https://docs.github.com/en/actions/reference/workflows-and-actions/workflow-syntax#jobsjob_idruns-on)、[工作流步骤与命令](https://docs.github.com/en/actions/reference/workflows-and-actions/workflow-syntax#jobsjob_idstepsrun)、[敏感值注入与输出边界](https://docs.github.com/en/actions/how-tos/write-workflows/choose-what-workflows-do/use-secrets#using-secrets-in-a-workflow)。 |